# ============================================
# 1. Import Required Libraries
# ============================================

In [6]:
import os
import cv2
import numpy as np
import shutil
from glob import glob

# ------------------------------
# 2. Paths setup
# ------------------------------

In [ ]:
benign_dir = r"C:\Users\Mohammadhossein\PycharmProjects\thesis"
malignant_dir = r"C:\Users\Mohammadhossein\PycharmProjects\thesist"
output_dir = r"C:\Users\Mohammadhossein\PycharmProjects\thesis\Make_Dataset"

# Create output folders if not exist
os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, "images"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "masks"), exist_ok=True)

# ------------------------------
# 3. Helper function to merge multiple masks
# ------------------------------

In [ ]:
def merge_masks(mask_paths):
    
    merged_mask = None
    ref_shape = None

    for mpath in mask_paths:
        mask = cv2.imread(mpath, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            print(f"[WARNING] Could not read mask: {mpath}")
            continue

        mask = (mask > 0).astype(np.uint8) * 255

        # Initialize reference shape from the first valid mask
        if ref_shape is None:
            ref_shape = mask.shape
            merged_mask = np.zeros(ref_shape, dtype=np.uint8)

        # Check consistency of shape
        if mask.shape != ref_shape:
            print(f"[WARNING] Mask {os.path.basename(mpath)} skipped (different size: {mask.shape} vs {ref_shape})")
            continue

        # Merge using bitwise OR (no resize)
        merged_mask = cv2.bitwise_or(merged_mask, mask)

    return merged_mask

# ------------------------------
# 4. Helper function to Process both benign and malignant classes
# ------------------------------

In [9]:
def process_class(class_dir, class_name, output_dir):
    print(f"\n[INFO] Processing class: {class_name}")

    # Get all non-mask images
    image_paths = sorted(glob(os.path.join(class_dir, "*.png")))
    image_paths = [p for p in image_paths if "_mask" not in os.path.basename(p)]

    for idx, img_path in enumerate(image_paths, start=1):
        base_name = os.path.splitext(os.path.basename(img_path))[0]

        # Find corresponding mask(s)
        mask_pattern = os.path.join(class_dir, base_name + "_mask*.png")
        mask_paths = glob(mask_pattern)

        if not mask_paths:
            print(f"[WARNING] No mask found for {base_name}, skipping.")
            continue

        merged_mask = merge_masks(mask_paths)
        if merged_mask is None:
            print(f"[WARNING] Could not merge masks for {base_name}.")
            continue

        # If multiple masks existed, remove old ones
        if len(mask_paths) > 1:
            for old_mask in mask_paths:
                try:
                    os.remove(old_mask)
                    print(f"[CLEANUP] Removed old mask: {os.path.basename(old_mask)}")
                except Exception as e:
                    print(f"[ERROR] Could not delete {old_mask}: {e}")

        # Save new merged mask (same size as originals)
        new_mask_path = os.path.join(class_dir, base_name + "_mask.png")
        cv2.imwrite(new_mask_path, merged_mask)
        print(f"[NEW MASK] Saved merged mask: {os.path.basename(new_mask_path)}")

        # Define output filenames for dataset
        img_out_name = f"{class_name}_{idx}.png"
        mask_out_name = f"{class_name}_{idx}_mask.png"

        img_out_path = os.path.join(output_dir, "images", img_out_name)
        mask_out_path = os.path.join(output_dir, "masks", mask_out_name)

        # Copy both without resizing
        shutil.copy(img_path, img_out_path)
        shutil.copy(new_mask_path, mask_out_path)

        print(f"[OK] Saved -> {img_out_name} & {mask_out_name}")

# ------------------------------
# Run processing
# ------------------------------

In [10]:
process_class(benign_dir, "benign", output_dir)
process_class(malignant_dir, "malignant", output_dir)

print("\n✅ All dataset prepared successfully!")
print(f"➡ Images saved to: {os.path.join(output_dir, 'images')}")
print(f"➡ Masks saved to:  {os.path.join(output_dir, 'masks')}")


[INFO] Processing class: benign
[NEW MASK] Saved merged mask: benign (1)_mask.png
[OK] Saved -> benign_1.png & benign_1_mask.png
[NEW MASK] Saved merged mask: benign (10)_mask.png
[OK] Saved -> benign_2.png & benign_2_mask.png
[CLEANUP] Removed old mask: benign (100)_mask.png
[CLEANUP] Removed old mask: benign (100)_mask_1.png
[NEW MASK] Saved merged mask: benign (100)_mask.png
[OK] Saved -> benign_3.png & benign_3_mask.png
[NEW MASK] Saved merged mask: benign (101)_mask.png
[OK] Saved -> benign_4.png & benign_4_mask.png
[NEW MASK] Saved merged mask: benign (102)_mask.png
[OK] Saved -> benign_5.png & benign_5_mask.png
[NEW MASK] Saved merged mask: benign (103)_mask.png
[OK] Saved -> benign_6.png & benign_6_mask.png
[NEW MASK] Saved merged mask: benign (104)_mask.png
[OK] Saved -> benign_7.png & benign_7_mask.png
[NEW MASK] Saved merged mask: benign (105)_mask.png
[OK] Saved -> benign_8.png & benign_8_mask.png
[NEW MASK] Saved merged mask: benign (106)_mask.png
[OK] Saved -> benign_9.p